# LLM 학습 파이프라인과 PEFT

대규모 언어 모델(Large Language Model, LLM)은 대규모 텍스트에서 언어 패턴을 익히는 **사전학습**, 목적에 맞는 예시로 행동을 조정하는 **파인튜닝**, 사람의 선호와 안전 기준에 맞추는 **정렬(Alignment)** 과정을 거친다. 이 노트북은 각 단계의 역할을 구분하고, 파인튜닝에서 전체 파라미터를 갱신하는 방법과 일부만 학습하는 PEFT를 비교한다.


## 1. 사전학습(Pre-training)

사전학습은 LLM이 웹 문서, 서적, 논문, 뉴스와 같은 대규모 비라벨 텍스트에서 언어의 구조와 일반 지식을 익히는 첫 단계이다. 사람이 문장마다 정답 라벨을 붙이지 않고 데이터 자체에서 학습 신호를 만들기 때문에 **자기지도학습(Self-supervised Learning)** 이라고 한다.

### 다음 토큰 예측

Decoder 계열 LLM은 앞의 토큰이 주어졌을 때 다음 토큰의 확률을 예측하는 자기회귀 언어 모델링을 주로 사용한다. 토큰 열 `w_1, ..., w_n`의 결합 확률은 조건부 확률의 곱으로 표현한다.

$$ P(w_1, w_2, ..., w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, ..., w_{i-1}) $$

여기서 `w_i`는 `i`번째 토큰이며, 오른쪽 조건은 그보다 앞에 있는 토큰이다. 모델은 각 위치의 예측 확률과 실제 다음 토큰 사이의 **교차 엔트로피 손실(Cross-entropy Loss)**을 줄이도록 파라미터를 갱신한다.

### 데이터와 Transformer

사전학습 데이터에는 Common Crawl과 같은 웹 자료, 서적, 논문, 뉴스, 위키피디아 등이 사용된다. Transformer의 Self-Attention은 문장 안의 토큰 관계를 계산하고, 순환 구조 없이 여러 토큰을 병렬 처리할 수 있어 대규모 학습에 적합하다. 사전학습이 끝나면 모델은 문법, 문맥과 일반 지식을 넓게 갖추지만 특정 업무의 답변 형식이나 조직 규칙까지 자동으로 따르지는 않는다.


## 2. 파인튜닝(Fine-tuning)

파인튜닝은 사전학습 모델에 특정 과업이나 도메인의 데이터를 추가로 학습시켜 응답 행동, 출력 형식 또는 도메인 적합성을 조정하는 단계이다. <br>
대표적인 **SFT(Supervised Fine-Tuning)** 는 `입력 → 기대 assistant 답변` 쌍에서 다음 토큰 손실을 줄이는 지도 학습 방식이다.

SFT는 **무엇을 정답으로 학습할지**를 정하고, Full Fine-tuning과 PEFT는 그 손실을 줄일 때 **어떤 파라미터를 갱신할지**를 정한다. 따라서 `LoRA로 SFT한다`는 표현은 SFT 학습 방식에 LoRA 갱신 전략을 적용한다는 뜻이다.

- **Full Fine-tuning**은 base model의 학습 대상 파라미터 전체를 갱신한다. 높은 자유도를 갖지만 가중치, gradient와 optimizer state를 위한 큰 GPU 메모리가 필요하다.
- **PEFT(Parameter-Efficient Fine-Tuning)** 는 base model 대부분을 고정하고 adapter 또는 prefix처럼 일부 파라미터만 학습한다. 학습 메모리와 checkpoint 크기를 줄일 수 있지만 Full Fine-tuning과 품질이 항상 같다고 보장되지는 않는다.

### 문제에 맞는 방법 선택하기

아래 그림은 모델의 부족한 부분을 먼저 진단하는 순서이다. Prompt Engineering과 RAG는 모델 파라미터를 바꾸지 않으며, 반복되는 행동이나 출력 형식을 학습해야 할 때 SFT를 검토한다.

```mermaid
flowchart LR
    A["문제 확인"] --> B{"무엇이 부족한가?"}
    B -->|지시가 불명확함| C["Prompt Engineering"]
    B -->|최신·외부 지식이 필요함| D["RAG"]
    B -->|행동·형식을 반복 학습함| E["SFT"]
    E --> F{"파라미터 갱신 범위"}
    F -->|전체 갱신| G["Full Fine-tuning"]
    F -->|일부만 갱신| H["PEFT"]
```

세 방법은 서로 배타적인 대안이 아니다. 예를 들어 SFT로 응답 형식을 학습하고 RAG로 최신 근거를 제공하며 Prompt Engineering으로 요청 조건을 명확히 하는 방식으로 함께 사용할 수 있다.

파인튜닝은 일반적으로 base model과 tokenizer 선택, 데이터 준비, 학습 전략과 하이퍼파라미터 설정, 학습, 평가 순서로 진행한다. 학습 loss만 보지 않고 별도로 분리한 평가셋에서 형식 준수, 정확도와 안전성을 학습 전 baseline과 비교해야 한다.


# PEFT(Parameter-Efficient Fine-Tuning)

PEFT는 대형 사전학습 모델의 대부분 파라미터를 고정한 채 일부 파라미터만 추가하거나 선택하여 학습하는 방법의 모음이다. 전체 모델 사본 대신 작은 adapter만 저장·교체할 수 있어 GPU 메모리, 학습 시간과 저장 공간을 줄이는 데 유용하다. 절감 비율은 모델과 기법에 따라 달라지므로 특정 비율을 항상 보장하지는 않는다.

주요 계열은 다음과 같다.

| 범주 | 대표 기법 | 학습하는 대상 | 특징 |
| --- | --- | --- | --- |
| Additive | Adapter, Prompt Tuning, Prefix Tuning | 새 모듈·연속 벡터 | base model을 보존하지만 추론 연산이 늘 수 있다. |
| Selective | BitFit, 일부 layer 학습 | 기존 파라미터 일부 | 추가 모듈은 없지만 학습 대상을 선택해야 한다. |
| Reparameterization | LoRA | 저랭크 가중치 변화량 | 작게 학습하고 base weight에 병합할 수 있다. |
| Quantized PEFT | QLoRA | 양자화 base와 LoRA | 메모리를 더 줄이지만 하드웨어 호환성을 확인해야 한다. |

Prompt Tuning은 입력 앞에 붙는 학습 가능한 soft prompt를 최적화한다. Prefix Tuning은 각 Transformer layer의 attention이 참조할 prefix key/value를 학습하므로 주입 위치가 다르다. LoRA는 다음 노트북에서 `W`, `A`, `B`의 행렬 연산으로 직접 확인한다.

공식 문서: [Hugging Face PEFT](https://huggingface.co/docs/peft/index)
